In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="FR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.20783598721027374
epoch:  1, loss: 0.12581504881381989
epoch:  2, loss: 0.06650485098361969
epoch:  3, loss: 0.0484713539481163
epoch:  4, loss: 0.04055729880928993
epoch:  5, loss: 0.03714336082339287
epoch:  6, loss: 0.035710401833057404
epoch:  7, loss: 0.03509480506181717
epoch:  8, loss: 0.03482012450695038
epoch:  9, loss: 0.03468785062432289
epoch:  10, loss: 0.03461708500981331
epoch:  11, loss: 0.03457321226596832
epoch:  12, loss: 0.034522633999586105
epoch:  13, loss: 0.0344557948410511
epoch:  14, loss: 0.03438945859670639
epoch:  15, loss: 0.034353163093328476
epoch:  16, loss: 0.03423091769218445
epoch:  17, loss: 0.034151311963796616
epoch:  18, loss: 0.03407735005021095
epoch:  19, loss: 0.034006066620349884
epoch:  20, loss: 0.03397497534751892
epoch:  21, loss: 0.033911533653736115
epoch:  22, loss: 0.033796824514865875
epoch:  23, loss: 0.03372236713767052
epoch:  24, loss: 0.03371824324131012
epoch:  25, loss: 0.0335865281522274
epoch:  26, loss: 

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7270112037658691
Test metrics:  R2 = 0.6985198259353638


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2989048361778259
epoch:  1, loss: 0.1851406991481781
epoch:  2, loss: 0.11822410672903061
epoch:  3, loss: 0.062456339597702026
epoch:  4, loss: 0.04270656406879425
epoch:  5, loss: 0.03813204541802406
epoch:  6, loss: 0.0360005758702755
epoch:  7, loss: 0.035425201058387756
epoch:  8, loss: 0.035168085247278214
epoch:  9, loss: 0.03510117530822754
epoch:  10, loss: 0.035070884972810745
epoch:  11, loss: 0.03506293147802353
epoch:  12, loss: 0.03505925089120865
epoch:  13, loss: 0.03505827859044075
epoch:  14, loss: 0.03505774587392807
epoch:  15, loss: 0.03505759686231613
epoch:  16, loss: 0.0350574254989624
epoch:  17, loss: 0.03505733981728554
epoch:  18, loss: 0.03505711629986763
epoch:  19, loss: 0.03505699709057808
epoch:  20, loss: 0.035056840628385544
epoch:  21, loss: 0.0350566990673542
epoch:  22, loss: 0.03505663946270943
epoch:  23, loss: 0.035056523978710175
epoch:  24, loss: 0.03505638614296913
epoch:  25, loss: 0.03505634889006615
epoch:  26, loss: 0.0

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7482489347457886
Test metrics:  R2 = 0.7200125455856323


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="PRP+"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3603939712047577
epoch:  1, loss: 0.1998813897371292
epoch:  2, loss: 0.11879340559244156
epoch:  3, loss: 0.07754289358854294
epoch:  4, loss: 0.05654958635568619
epoch:  5, loss: 0.04582025855779648
epoch:  6, loss: 0.040407031774520874
epoch:  7, loss: 0.03770666942000389
epoch:  8, loss: 0.03637111932039261
epoch:  9, loss: 0.03571413457393646
epoch:  10, loss: 0.035391297191381454
epoch:  11, loss: 0.0352318100631237
epoch:  12, loss: 0.035151828080415726
epoch:  13, loss: 0.035110484808683395
epoch:  14, loss: 0.03508791700005531
epoch:  15, loss: 0.03507453203201294
epoch:  16, loss: 0.03505510091781616
epoch:  17, loss: 0.035011496394872665
epoch:  18, loss: 0.034998539835214615
epoch:  19, loss: 0.03497349098324776
epoch:  20, loss: 0.034959133714437485
epoch:  21, loss: 0.034941885620355606
epoch:  22, loss: 0.03492873162031174
epoch:  23, loss: 0.034908127039670944
epoch:  24, loss: 0.03488772734999657
epoch:  25, loss: 0.034876491874456406
epoch:  26, los

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7688912153244019
Test metrics:  R2 = 0.7497383952140808


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="HS"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.054645951837301254
epoch:  1, loss: 0.04520779103040695
epoch:  2, loss: 0.03594518452882767
epoch:  3, loss: 0.035768430680036545
epoch:  4, loss: 0.035692401230335236
epoch:  5, loss: 0.03561607375741005
epoch:  6, loss: 0.03557198867201805
epoch:  7, loss: 0.03548542410135269
epoch:  8, loss: 0.035343315452337265
epoch:  9, loss: 0.03526611626148224
epoch:  10, loss: 0.035186268389225006
epoch:  11, loss: 0.03513500466942787
epoch:  12, loss: 0.03507734835147858
epoch:  13, loss: 0.0349179282784462
epoch:  14, loss: 0.03488236665725708
epoch:  15, loss: 0.03481096774339676
epoch:  16, loss: 0.03466382995247841
epoch:  17, loss: 0.03464075177907944
epoch:  18, loss: 0.03457536920905113
epoch:  19, loss: 0.034340906888246536
epoch:  20, loss: 0.03424805775284767
epoch:  21, loss: 0.03417908027768135
epoch:  22, loss: 0.03412957489490509
epoch:  23, loss: 0.03404119983315468
epoch:  24, loss: 0.03404119983315468
epoch:  25, loss: 0.03394864499568939
epoch:  26, loss:

In [12]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7313398122787476
Test metrics:  R2 = 0.7037297487258911


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="DY"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.05225568637251854
epoch:  1, loss: 0.04323606565594673
epoch:  2, loss: 0.04323606565594673
epoch:  3, loss: 0.036115627735853195
epoch:  4, loss: 0.03551451861858368
epoch:  5, loss: 0.03551451861858368
epoch:  6, loss: 0.03493865951895714
epoch:  7, loss: 0.034643467515707016
epoch:  8, loss: 0.03458518534898758
epoch:  9, loss: 0.03458518534898758
epoch:  10, loss: 0.033758971840143204
epoch:  11, loss: 0.03362897410988808
epoch:  12, loss: 0.03362897410988808
epoch:  13, loss: 0.03357728570699692
epoch:  14, loss: 0.033533867448568344
epoch:  15, loss: 0.033533867448568344
epoch:  16, loss: 0.03348080441355705
epoch:  17, loss: 0.03338145092129707
epoch:  18, loss: 0.03338145092129707
epoch:  19, loss: 0.03328521177172661
epoch:  20, loss: 0.03325527533888817
epoch:  21, loss: 0.03325527533888817
epoch:  22, loss: 0.033221933990716934
epoch:  23, loss: 0.03311772271990776
epoch:  24, loss: 0.03311772271990776
epoch:  25, loss: 0.03308480978012085
epoch:  26, loss

In [14]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8387651443481445
Test metrics:  R2 = 0.8328851461410522
